# Hearing Lens analytical-core review

Prepared October 1, 2026. This notebook is a reproducible review workspace; its code cells have not been executed in this artifact. The measured snapshot below comes from the saved command-line validation report. Human agreement/coherence review and the federal corpus remain pending.

| Corpus | Themes | First / repeat seconds | Quotes per theme |
|---|---:|---|---|
| Synthetic, 1,000 rows | 6 | 17.883 / 6.664 | 3,3,2,1,1,1 |
| CFPB, 100 rows | 6 | 18.188 / 17.243 | 3,3,1,2,2,1 |
| Federal | Pending | Pending | Pending |

Use only local development samples. Do not commit executed cells containing public narratives or review text. Missing data must remain marked pending.

In [ ]:
from pathlib import Path
import sys, json
import pandas as pd
import torch
from IPython.display import display, Markdown
ROOT = next(p for p in [Path.cwd(), Path.cwd().parent] if (p / "pipeline").is_dir())
sys.path.insert(0, str(ROOT))
from scripts.validate_cfpb_themes import load_sample
from scripts.validate_core import load_federal, run_corpus
from pipeline.timeline import timeline_figures
torch.set_num_threads(2)
CFPB_DIR = ROOT / "data/public_samples/cfpb_2017"  # Set to your prepared local sample.
FEDERAL_DIR = ROOT / "data/public_samples/federal"
OUTPUT_DIR = ROOT / "data/results/core-review"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
corpora = {"synthetic": pd.read_csv(ROOT / "data/samples/synthetic_chicago_hearing.csv")}
if (CFPB_DIR / "cfpb_manifest.json").exists():
    corpora["cfpb"] = load_sample(CFPB_DIR)[0].rename(columns={"date_received": "hearing_date"})
else:
    display(Markdown("**CFPB pending:** configure the prepared sample directory."))
if (FEDERAL_DIR / "federal_manifest.json").exists():
    corpora["federal"] = load_federal(FEDERAL_DIR)[0]
else:
    display(Markdown("**Federal pending:** a complete verified sample is required."))
summary = {name: run_corpus(frame, name, OUTPUT_DIR) for name, frame in corpora.items()}
display(pd.DataFrame(summary).T)


In [ ]:
for name in summary:
    packet = json.loads((OUTPUT_DIR / f"{name}-review.json").read_text())
    display(Markdown(f"## {name}"))
    for figure in timeline_figures(packet["result"]["timeline"]).values():
        figure.show()


## Human review remains separate

Review the local theme packets, label 20 comments per corpus without consulting predictions, and run the agreement evaluator described in `data/validation/README.md`. Do not report a gate passed because these cells run. Review every failed acceptance check and record Benjamin's actual decision.